In [2]:
import os
import sys
import glob
import pandas as pd
import numpy as np
from pathlib import Path
sys.path.append("..")

### 1. mix_ratio 敏感性分析

In [4]:
from process.utils import get_mix_ratio_results


root = Path.cwd().parent

mix_37235 = get_mix_ratio_results(root/'mix_ratio'/'mix_37_235')
mix_55532 = get_mix_ratio_results(root/'mix_ratio'/'mix_55_532')

### 2. 最终分类结果分析

In [6]:
from process.utils import get_all_clf_results
from latex_utils import generate_latex_table


all_results = get_all_clf_results(root)

summary_all = (all_results.groupby(["Dataset", "Method", "Classifier"])
    .agg(F1_mean=("F1_score", "mean"), F1_std=("F1_score", "std"),
        G_mean=("G_mean", "mean"), G_std=("G_mean", "std") ).reset_index()
)

IFC_METHODS = {"none_noft", "none_ft", "bootstrap_noft", "bootstrap_ft", "smote_noft", "smote_ft"}
BASELINE_METHODS = {"smote", "ctgan", "tvae", "forestdiffusion", "tabddpm", "tabkde", "raw"}
baseline_results = all_results[
    all_results["Method"].isin(BASELINE_METHODS)
]


final_method = "smote_ft"
final_stage = 2

ifc_final = all_results[
    (all_results["Method"] == final_method) & (all_results["stage_depth"] == final_stage)
]

compare_results = pd.concat(
    [baseline_results, ifc_final], ignore_index=True
)

compare_summary = (
    compare_results.groupby(["Dataset", "Method", "Classifier"])
    .agg(F1_mean=("F1_score", "mean"), F1_std=("F1_score", "std"),
         G_mean=("G_mean", "mean"), G_std=("G_mean", "std")).reset_index()
)

method_order =  ["raw", "smote", "ctgan", "tvae", "forestdiffusion", "tabddpm", "tabkde", "smote_ft"]

compare_summary["Method"] = pd.Categorical(
    compare_summary["Method"], categories=method_order, ordered=True
)

compare_summary = compare_summary.sort_values(["Dataset", "Method"])

latex = generate_latex_table(
    compare_summary,
    datasets=["framingham", "hepatitis"],
    methods=method_order
)
print(latex)

\begin{tabular}{llllllll}
\toprule
Dataset & Method & SVC_F1 & SVC_G & MLP_F1 & MLP_G & XGB_F1 & XGB_G \\
\midrule
framingham & raw & \textbf{0.38 $\pm$ .02} & \textbf{0.67 $\pm$ .02} & 0.06 $\pm$ .05 & 0.15 $\pm$ .11 & 0.06 $\pm$ .03 & 0.17 $\pm$ .05 \\
 & smote & 0.34 $\pm$ .03 & 0.63 $\pm$ .03 & 0.28 $\pm$ .03 & 0.53 $\pm$ .03 & 0.35 $\pm$ .02 & \underline{0.63 $\pm$ .02} \\
 & ctgan & 0.30 $\pm$ .04 & 0.52 $\pm$ .05 & 0.23 $\pm$ .05 & 0.40 $\pm$ .05 & 0.26 $\pm$ .05 & 0.45 $\pm$ .06 \\
 & tvae & 0.36 $\pm$ .02 & 0.65 $\pm$ .02 & 0.27 $\pm$ .03 & 0.51 $\pm$ .04 & 0.28 $\pm$ .03 & 0.53 $\pm$ .04 \\
 & forestdiffusion & 0.29 $\pm$ .04 & 0.48 $\pm$ .04 & 0.20 $\pm$ .03 & 0.36 $\pm$ .03 & 0.15 $\pm$ .04 & 0.30 $\pm$ .04 \\
 & tabddpm & 0.35 $\pm$ .02 & 0.63 $\pm$ .02 & \textbf{0.34 $\pm$ .03} & \underline{0.60 $\pm$ .03} & \underline{0.36 $\pm$ .03} & 0.61 $\pm$ .04 \\
 & tabkde & \underline{0.37 $\pm$ .02} & \underline{0.66 $\pm$ .02} & \underline{0.34 $\pm$ .02} & \textbf{0.62 $\pm$ .

In [29]:
ifc_results = all_results[all_results["Method"].isin(IFC_METHODS)].copy()

ifc_summary = (
    ifc_results.groupby(["Dataset", "Classifier", "Method", "stage_depth"])
    .agg(F1_mean=("F1_score","mean"), F1_std=("F1_score", "std"),
        G_mean=("G_mean","mean"), G_std=("G_mean", "std")).reset_index())

best_ifc_config = (
    ifc_summary.sort_values(["Dataset", "Classifier", "F1_mean"],ascending=[True, True, False])
    .groupby(["Dataset", "Classifier"]).first().reset_index())

best_ifc_results = all_results.merge(
    best_ifc_config[["Dataset", "Classifier", "Method", "stage_depth"]],
    on=["Dataset", "Classifier", "Method", "stage_depth"], how="inner"
)

best_ifc_results = best_ifc_results.copy()
best_ifc_results["Method_show"] = "IFC-HFlowVAE"
baseline_results["Method_show"] = baseline_results["Method"]
best_compare_results = pd.concat([baseline_results, best_ifc_results], ignore_index=True)


print(best_compare_results["Method_show"].unique())

final_summary = (
    best_compare_results.groupby(["Dataset", "Method_show", "Classifier"])
    .agg(F1_mean=("F1_score","mean"), F1_std=("F1_score","std"),
        G_mean=("G_mean","mean"), G_std=("G_mean","std")).reset_index()
)

latex_data = final_summary.copy()
latex_data["Method"] = latex_data["Method_show"]

print(latex_data["Method"].unique())
methods_order = ["raw","smote","ctgan","tvae","forestdiffusion","tabddpm","tabkde","IFC-HFlowVAE"]

latex_data["Method"] = pd.Categorical(
    latex_data["Method"], categories=methods_order, ordered=True
)
print(latex_data["Method"].unique())
latex_data = latex_data.sort_values(["Dataset", "Method"])

DATASETS = ['breast_cancer_coimbra', 'PIMA_diabetes','TCGA_InfoWithGrade', 'Thyroid_Diff',
            'diabetes_risk_prediction', 'framingham', 'hepatitis']

latex = generate_latex_table(
    latex_data,
    datasets=['TCGA_InfoWithGrade', 'Thyroid_Diff', 'diabetes_risk_prediction'],
    methods=methods_order
)
print(latex)

['raw' 'smote' 'ctgan' 'tvae' 'forestdiffusion' 'tabddpm' 'tabkde'
 'IFC-HFlowVAE']
['IFC-HFlowVAE' 'ctgan' 'forestdiffusion' 'raw' 'smote' 'tabddpm' 'tabkde'
 'tvae']
['IFC-HFlowVAE', 'ctgan', 'forestdiffusion', 'raw', 'smote', 'tabddpm', 'tabkde', 'tvae']
Categories (8, object): ['raw' < 'smote' < 'ctgan' < 'tvae' < 'forestdiffusion' < 'tabddpm' < 'tabkde' < 'IFC-HFlowVAE']
\begin{tabular}{llllllll}
\toprule
Dataset & Method & SVC_F1 & SVC_G & MLP_F1 & MLP_G & XGB_F1 & XGB_G \\
\midrule
TCGA_InfoWithGrade & raw & \underline{0.86 $\pm$ .03} & \underline{0.88 $\pm$ .03} & 0.83 $\pm$ .02 & 0.86 $\pm$ .02 & 0.85 $\pm$ .02 & 0.87 $\pm$ .02 \\
 & smote & 0.86 $\pm$ .03 & 0.88 $\pm$ .02 & 0.85 $\pm$ .03 & 0.87 $\pm$ .03 & \textbf{0.86 $\pm$ .03} & \textbf{0.88 $\pm$ .02} \\
 & ctgan & 0.86 $\pm$ .03 & 0.88 $\pm$ .02 & 0.84 $\pm$ .03 & 0.86 $\pm$ .03 & 0.85 $\pm$ .03 & 0.87 $\pm$ .02 \\
 & tvae & 0.86 $\pm$ .03 & 0.88 $\pm$ .03 & 0.84 $\pm$ .03 & 0.86 $\pm$ .03 & 0.86 $\pm$ .03 & 0.87 $\pm$ 

C:\Users\HC\AppData\Local\Temp\ipykernel_24728\4154252328.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  baseline_results["Method_show"] = baseline_results["Method"]


In [9]:
print(baseline_results["Method"].unique())
print(best_ifc_results["Method"].unique())

['raw' 'smote' 'ctgan' 'tvae' 'forestdiffusion' 'tabddpm' 'tabkde']
['bootstrap_noft' 'smote_ft' 'none_noft' 'none_ft' 'smote_noft']


In [30]:
smote_stage3 = all_results[
    (all_results["Method"]=="smote_ft") &
    (all_results["stage_depth"]==3)
]

ifc_summary = (
    smote_stage3
    .groupby(
        [
            "Dataset",
            "Classifier",
            "Method",
        ]
    )
    .agg(
        F1_mean=("F1_score","mean"),
        G_mean=("G_mean","mean")
    )
    .reset_index()
)

best_smote3_config = (
    ifc_summary
    .sort_values(
        [
            "Dataset",
            "Classifier",
            "F1_mean"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .groupby(
        [
            "Dataset",
            "Classifier"
        ]
    )
    .first()
    .reset_index()
)


In [14]:
df = all_results
main_ablation = df[df["Method"].isin(IFC_METHODS)]

main_ablation = main_ablation[(main_ablation["stage_depth"] == 3)]

main_ablation["config"] = main_ablation["prior_type"] + "_" + main_ablation["finetune"].map({True:"ft", False: "noft"})

per_block = (main_ablation.groupby(["Dataset", "Classifier", "config"])
             .agg(mean_f1=("F1_score", "mean")).reset_index())
per_block["rank"] = per_block.groupby(["Dataset", "Classifier"])["mean_f1"].rank(ascending=False)

In [ ]:
baseline_results = all_results[all_results["Method"].isin(BASELINE_METHODS)].copy()

baseline_summary = (
    baseline_results
    .groupby(["Dataset", "Method", "Classifier"])
    .agg(
        F1_mean=("F1_score", "mean"),
        F1_std=("F1_score", "std"),
        G_mean=("G_mean", "mean"),
        G_std=("G_mean", "std")
    )
    .reset_index()
)

In [31]:


ifc_results = all_results[all_results["Method"].isin(IFC_METHODS)].copy()

ifc_config_summary = (
    ifc_results.groupby(["Dataset", "Classifier", "Method", "prior_type", "finetune", "stage_depth"]).agg(
        F1_mean=("F1_score", "mean"),
        F1_std=("F1_score", "std"),
        G_mean=("G_mean", "mean"),
        G_std = ("G_mean", "std")
    )
)


In [16]:
datasets_show = ['breast_cancer_coimbra', 'PIMA_diabetes','TCGA_InfoWithGrade', 'Thyroid_Diff',
            'diabetes_risk_prediction', 'framingham', 'hepatitis']

methods_show = ["raw", "smote", "ctgan", "tvae", "forestdiffusion", "tabddpm", "tabkde"]

latex = generate_latex_table(
    baseline_summary,
    datasets=datasets_show,
    methods=methods_show
)

print(latex)

\begin{tabular}{llllllll}
\toprule
Dataset & Method & SVC_F1 & SVC_G & MLP_F1 & MLP_G & XGB_F1 & XGB_G \\
\midrule
breast_cancer_coimbra & ctgan & 0.60 $\pm$ .13 & 0.61 $\pm$ .10 & 0.69 $\pm$ .08 & 0.36 $\pm$ .21 & 0.76 $\pm$ .04 & 0.68 $\pm$ .07 \\
 & forestdiffusion & 0.62 $\pm$ .10 & 0.61 $\pm$ .09 & \textbf{0.70 $\pm$ .05} & 0.34 $\pm$ .20 & 0.73 $\pm$ .05 & 0.66 $\pm$ .07 \\
 & raw & \underline{0.70 $\pm$ .08} & \underline{0.67 $\pm$ .08} & 0.56 $\pm$ .19 & \textbf{0.50 $\pm$ .13} & 0.73 $\pm$ .05 & 0.70 $\pm$ .08 \\
 & smote & \textbf{0.70 $\pm$ .07} & 0.67 $\pm$ .08 & \underline{0.69 $\pm$ .07} & 0.42 $\pm$ .17 & \underline{0.78 $\pm$ .06} & \textbf{0.70 $\pm$ .09} \\
 & tabddpm & 0.66 $\pm$ .08 & 0.65 $\pm$ .08 & 0.69 $\pm$ .06 & 0.40 $\pm$ .18 & 0.75 $\pm$ .07 & 0.67 $\pm$ .10 \\
 & tabkde & 0.70 $\pm$ .08 & \textbf{0.68 $\pm$ .07} & 0.69 $\pm$ .06 & 0.40 $\pm$ .18 & 0.75 $\pm$ .05 & 0.67 $\pm$ .07 \\
 & tvae & 0.69 $\pm$ .07 & 0.67 $\pm$ .07 & 0.69 $\pm$ .06 & \underline{0.48

In [17]:
datasets_show = ['breast_cancer_coimbra', 'PIMA_diabetes','TCGA_InfoWithGrade', 'Thyroid_Diff',
            'diabetes_risk_prediction', 'framingham', 'hepatitis']

methods_show = ["none_noft", "none_ft", "bootstrap_noft", "bootstrap_ft", "smote_noft", "smote_ft"]

latex = generate_latex_table(
    baseline_summary,
    datasets=datasets_show,
    methods=methods_show
)

print(latex)

\begin{tabular}{}
\toprule
\midrule
\bottomrule
\end{tabular}



In [ ]:
# 对比不同stage_depth下的结果均值

stage_methods = {
    "bootstrap_noft",
    "bootstrap_ft",
    "smote_noft",
    "smote_ft"
}

stage_results = all_results[
    all_results["Method"].isin(stage_methods)
].copy()

stage_summary = (
    stage_results
    .groupby(
        [
            "Dataset",
            "Method",
            "stage_depth",
            "Classifier"
        ]
    )
    .agg(
        F1_mean=("F1_score", "mean"),
        F1_std=("F1_score", "std"),
        G_mean=("G_mean", "mean"),
        G_std=("G_mean", "std")
    )
    .reset_index()
)

In [23]:
stage_compare = (
    stage_summary
    .pivot_table(
        index=[
            "Dataset",
            "Method",
            "Classifier"
        ],
        columns="stage_depth",
        values=[
            "F1_mean",
            "G_mean"
        ]
    )
    .reset_index()
)

In [54]:
stage_compare[stage_compare['Dataset']=="PIMA_diabetes"]

Dataset          Method Classifier   F1_mean            \
stage_depth                                                 2.0       3.0   
0            PIMA_diabetes    bootstrap_ft        MLP  0.636376  0.639174   
1            PIMA_diabetes    bootstrap_ft        SVC  0.659028  0.651410   
2            PIMA_diabetes    bootstrap_ft        XGB  0.669362  0.670785   
3            PIMA_diabetes  bootstrap_noft        MLP  0.638514  0.633851   
4            PIMA_diabetes  bootstrap_noft        SVC  0.661259  0.652405   
5            PIMA_diabetes  bootstrap_noft        XGB  0.674351  0.663910   
6            PIMA_diabetes        smote_ft        MLP  0.633104  0.635624   
7            PIMA_diabetes        smote_ft        SVC  0.655682  0.654344   
8            PIMA_diabetes        smote_ft        XGB  0.675173  0.674264   
9            PIMA_diabetes      smote_noft        MLP  0.636502  0.639593   
10           PIMA_diabetes      smote_noft        SVC  0.656427  0.654197   
11           PIMA_diabetes      smote_noft        XGB  0.671599  0.667603   

               G_mean            
stage_depth       2.0       3.0  
0            0.710937  0.711772  
1            0.733302  0.726493  
2            0.741753  0.742958  
3            0.710053  0.707203  
4            0.735202  0.727710  
5            0.746753  0.736777  
6            0.703190  0.709906  
7            0.730549  0.729723  
8            0.747347  0.746227  
9            0.710340  0.713957  
10           0.731212  0.729838  
11           0.744501  0.740672

In [41]:
stage2 = stage_summary[
    stage_summary["stage_depth"] == 2
].copy()

stage3 = stage_summary[
    stage_summary["stage_depth"] == 3
].copy()


compare = stage2.merge(
    stage3,
    on=[
        "Dataset",
        "Method",
        "Classifier"
    ],
    suffixes=("_stage2", "_stage3")
)

compare["F1_diff"] = (
    compare["F1_mean_stage3"] -
    compare["F1_mean_stage2"]
)

compare["G_diff"] = (
    compare["G_mean_stage3"] -
    compare["G_mean_stage2"]
)

compare[
    [
        "Dataset",
        "Method",
        "Classifier",
        "F1_diff",
        "G_diff"
    ]
]

,Dataset,Method,Classifier,F1_diff,G_diff
0,PIMA_diabetes,bootstrap_ft,MLP,0.002798,0.000835
1,PIMA_diabetes,bootstrap_ft,SVC,-0.007618,-0.006809
2,PIMA_diabetes,bootstrap_ft,XGB,0.001423,0.001205
3,PIMA_diabetes,bootstrap_noft,MLP,-0.004663,-0.002850
4,PIMA_diabetes,bootstrap_noft,SVC,-0.008854,-0.007492
...,...,...,...,...,...
79,hepatitis,smote_ft,SVC,-0.001128,0.000431
80,hepatitis,smote_ft,XGB,-0.014865,-0.012420
81,hepatitis,smote_noft,MLP,-0.032069,-0.014106
82,hepatitis,smote_noft,SVC,0.004543,0.005916


In [55]:
finetune_compare = (
    compare
    .copy()
)

finetune_compare["prior_type"] = (
    finetune_compare["Method"]
    .str.split("_")
    .str[0]
)

finetune_compare["finetune"] = (
    finetune_compare["Method"]
    .str.split("_")
    .str[1]
)


effect = (
    finetune_compare
    .groupby(
        [
            "prior_type",
            "finetune"
        ]
    )
    [
        [
            "F1_diff",
            "G_diff"
        ]
    ]
    .mean()
)

effect

F1_diff    G_diff
prior_type finetune                    
bootstrap  ft        0.002753  0.003697
           noft      0.000190  0.000224
smote      ft       -0.004127 -0.003302
           noft     -0.003694 -0.001842

In [47]:
method_effect = (
    compare
    .groupby("Method")
    [["F1_diff", "G_diff"]]
    .mean()
)

method_effect

,F1_diff,G_diff
Method,,
bootstrap_ft,0.002753,0.003697
bootstrap_noft,0.000190,0.000224
smote_ft,-0.004127,-0.003302
smote_noft,-0.003694,-0.001842


In [43]:
compare["F1_better"] = (
    compare["F1_diff"] > 0
)

compare["G_better"] = (
    compare["G_diff"] > 0
)


win_rate = (
    compare
    .groupby("Method")
    [["F1_better", "G_better"]]
    .mean()
)

win_rate

,F1_better,G_better
Method,,
bootstrap_ft,0.619048,0.619048
bootstrap_noft,0.476190,0.380952
smote_ft,0.238095,0.238095
smote_noft,0.238095,0.285714


In [44]:
from scipy.stats import wilcoxon


stat, p = wilcoxon(
    compare["F1_mean_stage2"],
    compare["F1_mean_stage3"]
)

print(stat, p)

1297.0 0.029527510097281426


In [45]:
from scipy.stats import wilcoxon

for method in compare["Method"].unique():
    sub = compare[compare["Method"] == method]

    stat, p = wilcoxon(
        sub["F1_mean_stage2"],
        sub["F1_mean_stage3"]
    )

    print(method, stat, p)

for method in compare["Method"].unique():
    sub = compare[compare["Method"] == method]

    stat, p = wilcoxon(
        sub["G_mean_stage2"],
        sub["G_mean_stage3"]
    )

    print(method, stat, p)

bootstrap_ft 81.0 0.2428417205810547
bootstrap_noft 96.0 0.5167608261108398
smote_ft 38.0 0.005541801452636719
smote_noft 54.0 0.03191947937011719
bootstrap_ft 79.0 0.21568012237548828
bootstrap_noft 81.0 0.2428417205810547
smote_ft 43.0 0.010125160217285156
smote_noft 70.0 0.11934185028076172


In [53]:
stage_prior = stage_summary.copy()

stage_prior = stage_prior[
    stage_prior["Method"].isin(
        [
            "bootstrap_noft",
            "bootstrap_ft",
            "smote_noft",
            "smote_ft"
        ]
    )
].copy()


stage_prior["prior_type"] = (
    stage_prior["Method"]
    .str.split("_")
    .str[0]
)

prior_stage_summary = (
    stage_prior
    .groupby(
        [
            "Dataset",
            "prior_type",
            "stage_depth",
            "Classifier"
        ]
    )
    .agg(
        F1_mean=("F1_mean", "mean"),
        G_mean=("G_mean", "mean")
    )
    .reset_index()
)

prior_stage2 = prior_stage_summary[
    prior_stage_summary["stage_depth"] == 2
].copy()


prior_stage3 = prior_stage_summary[
    prior_stage_summary["stage_depth"] == 3
].copy()


prior_compare = prior_stage2.merge(
    prior_stage3,
    on=[
        "Dataset",
        "prior_type",
        "Classifier"
    ],
    suffixes=("_stage2", "_stage3")
)

prior_compare["F1_diff"] = (
    prior_compare["F1_mean_stage3"]
    -
    prior_compare["F1_mean_stage2"]
)


prior_compare["G_diff"] = (
    prior_compare["G_mean_stage3"]
    -
    prior_compare["G_mean_stage2"]
)

prior_effect = (
    prior_compare
    .groupby("prior_type")
    [
        [
            "F1_diff",
            "G_diff"
        ]
    ]
    .mean()
)

prior_effect

prior_compare["F1_better"] = (
    prior_compare["F1_diff"] > 0
)

prior_compare["G_better"] = (
    prior_compare["G_diff"] > 0
)


prior_win_rate = (
    prior_compare
    .groupby("prior_type")
    [
        [
            "F1_better",
            "G_better"
        ]
    ]
    .mean()
)

prior_win_rate

from scipy.stats import wilcoxon


for prior in prior_compare["prior_type"].unique():

    sub = prior_compare[
        prior_compare["prior_type"] == prior
    ]

    print("\n", prior)

    for metric in [
        "F1_mean",
        "G_mean"
    ]:
        stat, p = wilcoxon(
            sub[f"{metric}_stage2"],
            sub[f"{metric}_stage3"]
        )

        print(
            metric,
            "stat=",
            stat,
            "p=",
            p
        )


 bootstrap
F1_mean stat= 108.0 p= 0.8116779327392578
G_mean stat= 110.0 p= 0.8648872375488281

 smote
F1_mean stat= 43.0 p= 0.010125160217285156
G_mean stat= 58.0 p= 0.04599285125732422


### 3. 读取真实和生成数据目录

In [5]:
from process.utils import get_all_manifest, build_fold_data_manifest

manifest = get_all_manifest(root)
fold_data_manifest = build_fold_data_manifest(root/'artifacts')

manifest.head()

,Dataset,Seed,Fold,Method,fidelity_path,final_path,prior_type,finetune,stage_depth
0,breast_cancer_coimbra,1024,1,bootstrap_ft,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,bootstrap,True,3.0
1,breast_cancer_coimbra,1024,1,bootstrap_noft,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,bootstrap,False,3.0
2,breast_cancer_coimbra,1024,1,ctgan,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,NaN,NaN,NaN
3,breast_cancer_coimbra,1024,1,forestdiffusion,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,NaN,NaN,NaN
4,breast_cancer_coimbra,1024,1,none_ft,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,none,True,3.0


In [6]:
fold_data_manifest

,Dataset,Seed,Fold,X_train_path,X_valid_path,X_test_path,y_train_path,y_valid_path,y_test_path
0,breast_cancer_coimbra,1024,1,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...
1,breast_cancer_coimbra,1024,2,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...
2,breast_cancer_coimbra,1024,3,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...
3,breast_cancer_coimbra,2026,1,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...
4,breast_cancer_coimbra,2026,2,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...,e:\hflow_results_analysis\artifacts\breast_can...
...,...,...,...,...,...,...,...,...,...
150,Thyroid_Diff,456,1,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...
151,Thyroid_Diff,456,2,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...
152,Thyroid_Diff,456,3,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...
153,Thyroid_Diff,456,4,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...,e:\hflow_results_analysis\artifacts\Thyroid_Di...


In [7]:
# 第一步：只在 seed/fold 内聚合（这一步合理，seed/fold是重复测量，不是异质性来源）
per_block = (
    all_results[all_results["prior_type"].notna()]
    .groupby(["Dataset", "Classifier", "prior_type", "finetune", "stage_depth"])
    .agg(mean_f1=("F1_score", "mean"))
    .reset_index()
)

# 第二步：在每个 dataset×classifier 区组内部对各配置排名，
# 而不是跨数据集直接平均原始F1——这一步消除了数据集间量纲差异
per_block["rank"] = per_block.groupby(["Dataset", "Classifier"])["mean_f1"].rank(ascending=False)

ablation_rank_summary = (
    per_block.groupby(["prior_type", "finetune", "stage_depth"])
    .agg(mean_rank=("rank", "mean"), std_rank=("rank", "std"))
    .reset_index()
    .sort_values("mean_rank")
)

In [8]:
ablation_rank_summary

,prior_type,finetune,stage_depth,mean_rank,std_rank
4,none,False,3.0,4.000000,3.255764
8,smote,True,2.0,4.285714,2.239260
5,none,True,3.0,4.761905,2.913842
3,bootstrap,True,3.0,5.095238,2.879319
6,smote,False,2.0,5.095238,2.981690
0,bootstrap,False,2.0,5.666667,2.921187
9,smote,True,3.0,6.000000,2.323790
2,bootstrap,True,2.0,6.142857,2.495711
1,bootstrap,False,3.0,6.809524,2.856905
7,smote,False,3.0,7.142857,2.613154
